# 인공 신경망

keras 와 tensorflow 설정

* <mark>실습 결과가 실행할 때마다 달라지는 것을 줄이기 위한 설정</mark>

In [ ]:
# 실행마다 동일한 결과를 얻기 위해 케라스에 랜덤 시드를 사용하고 텐서플로 연산을 결정적으로 만듭니다.
import keras
import tensorflow as tf

# 과거에는 각각 설정했으나
# random.seed(42)
# np.random.seed(42)
# tf.random.set_seed(42)

# 현재는 위 3개를 한번에 설정함
keras.utils.set_random_seed(42)  # 42를 사용하는 것은 '관례'

# GPU 설정 --> "가능하면 매번 같은 순서와 같은 방식으로 계산해 주세요."
tf.config.experimental.enable_op_determinism()

## 패션 MNIST

* 패션 MNIST 란?  --> 이미지(data)를 보고 어떤 class 인지 맞히는 데이터셋
* 사진자체는 정답이 아닙니다 --> 사진은 28 x 28 data 픽셀
* 정답은 0 ~ 9 입니다

In [ ]:
import keras

(train_input, train_target), (test_input, test_target) = \
    keras.datasets.fashion_mnist.load_data()

In [ ]:
print(train_input.shape, train_target.shape)
train_input[0]   # 첫번째 이미지

In [ ]:
print(test_input.shape, test_target.shape)

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(10, 10, figsize=(10,10))   # subplots --> 10행 10열 , figure (전체 그림판) 각 1개의 figsize 10 x 10

# 10 × 10 배열을 일렬로 펼치기
axs = axs.ravel()

for i in range(100):
    axs[i].imshow(train_input[i], cmap='gray')  # 원본 이미지로 그대로 보여줍니다
    axs[i].axis('off')
plt.show()

In [ ]:
print(train_target[:10])    # 처음부터 10개
print(train_target[90:100])
class_names = ['티셔츠', '바지', '스웨터', '드레스', '코트', '샌들', '셔츠', '운동화', '가방', '부츠']

In [ ]:
import numpy as np

print(np.unique(train_target, return_counts=True))

## 로지스틱 회귀로 패션 아이템 분류하기

### 픽셀 값을 '0 ~ 255'에서 '0 ~ 1' 사이로 바꾸는 것입니다.

In [ ]:
train_scaled = train_input / 255
train_scaled = train_scaled.reshape(-1, 28*28)

In [ ]:
print(train_scaled.shape)
# print(train_scaled[0])

### 먼저 로지스틱 회귀를 사용

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.linear_model import SGDClassifier

sc = SGDClassifier(loss='log_loss', max_iter=5, random_state=42)
scores = cross_validate(sc, train_scaled, train_target, n_jobs=-1)
print(np.mean(scores['test_score']))

## 인공신경망

### 텐서플로와 케라스

In [ ]:
import tensorflow as tf

In [ ]:
import keras

### 백엔드로 텐서플로 사용

In [ ]:
keras.config.backend()

### 만약 케라스의 백엔드를 바꾸려면?

* 환경 변수 KERAS_BACKEND”를 사용하세요
* 다만 벡엔드를 바꾸려면 케라스 패키지를 임포트하기  전에  "KERAS_BACKEND"  환경  변수를 설정해야 합니다
* 코랩의 경우에는 런타임을 종료하고 다시 시작해 주세요.

In [ ]:
# import os
# os.environ["KERAS_BACKEND"] = "torch"   # 또는 "jax"

## 인공신경망으로 모델 만들기

### <mark>ANN (인공신경망) 은 테스트 데이터와 검증 (val_scaled) 데이터 구분해 확인</mark>

In [ ]:
from sklearn.model_selection import train_test_split

train_scaled, val_scaled, train_target, val_target = train_test_split(
    train_scaled, train_target, test_size=0.2, random_state=42)

In [ ]:
print(train_scaled.shape, train_target.shape)

In [ ]:
print(val_scaled.shape, val_target.shape)

### 입력층 --> 28 x 28 --> 784 ea

In [ ]:
inputs = keras.layers.Input(shape=(784,))

### 밀집층 (Dense) -> 10 --> 클래스 10개 --> 소프트맥스 함수 --> 확률


In [ ]:
dense = keras.layers.Dense(10, activation='softmax')2

In [ ]:
model = keras.Sequential([inputs, dense])

## 인공신경망으로 패션 아이템 분류하기

In [ ]:
model.compile(loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
print(train_target[:10])

In [ ]:
model.fit(train_scaled, train_target, epochs=5)

In [ ]:
model.evaluate(val_scaled, val_target)